# Pitcher strikeout exploratory analysis

This notebook reads the canonical Level 1 `pitcher_games.parquet` artifact.
Reusable stabilization and reliability logic lives in `Python.reliability`;
model-ready features are produced separately by pipeline Levels 2 and 3.

**Leakage reminder:** `PA`, `K`, and other same-game outcomes are valid here for
EDA and reliability analysis only. They must never enter the pregame feature set.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew

# --- Plot rendering toggle (keeps this .ipynb small so it doesn't bloat AI context) ---
# With the non-interactive "Agg" backend, figures are still written to the Graphs/
# folder by plt.savefig(...) but are NOT embedded inline, so no heavy base64 image
# outputs get stored in the notebook. plt.show() becomes a harmless no-op.
# To view plots inline instead: comment out the next line, then restart the kernel.
plt.switch_backend("Agg")

# Locate repo root (folder containing src/Python) to import the package.
_root = Path.cwd().resolve()
while _root != _root.parent and not (_root / "src" / "Python").exists():
    _root = _root.parent
if (_root / "src").exists() and str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from Python.config import PITCHER_GAMES_PATH
from Python import reliability as rel

# Graphs go next to this notebook, so renaming the folder never breaks the path.
try:
    NB_DIR = Path(__vsc_ipynb_file__).resolve().parent  # Cursor/VSCode
except NameError:
    NB_DIR = Path.cwd()
GRAPHS = NB_DIR / "Graphs"
GRAPHS.mkdir(parents=True, exist_ok=True)
print("Data:", PITCHER_GAMES_PATH)
print("Graphs ->", GRAPHS)

Data: C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\Data\processed\pitcher_games.parquet
Graphs -> C:\Users\ckaplinger\Downloads\Personal-Projects\MLB-Props\Models\Strikeout-Model\Strikeout-EDA\Graphs


In [2]:
games = pl.read_parquet(PITCHER_GAMES_PATH)
df = games.to_pandas()
print("rows, cols:", df.shape)
df.head()

rows, cols: (14124, 137)


,game_pk,pitcher,game_date,season,player_name,home_team,away_team,p_throws,Pitches,Strikes,...,contact_rate,zcontact_rate,ocontact_rate,swstr_rate,whiff_rate,lg_hr_fb_prior,FIP,xFIP,k_rate,pitcher_name
0,718774,645261,2023-03-30,2023,"Alcantara, Sandy",MIA,NYM,R,97,41,...,0.702128,0.750000,0.600000,0.144330,0.297872,0.128152,4.666765,5.254754,0.086957,Sandy Alcantara
1,718767,669456,2023-03-30,2023,"Bieber, Shane",SEA,CLE,R,87,37,...,0.761905,0.818182,0.555556,0.114943,0.238095,0.128152,2.255000,3.643309,0.130435,Shane Bieber
2,718777,669203,2023-03-30,2023,"Burnes, Corbin",CHC,MIL,R,87,39,...,0.780488,0.888889,0.571429,0.103448,0.219512,0.128152,4.380000,5.317108,0.130435,Corbin Burnes
3,718767,622491,2023-03-30,2023,"Castillo, Luis",SEA,CLE,R,76,46,...,0.704545,0.862069,0.400000,0.171053,0.295455,0.128152,1.255000,2.643309,0.315789,Luis Castillo
4,718768,656302,2023-03-30,2023,"Cease, Dylan",HOU,CWS,R,86,51,...,0.673913,0.846154,0.450000,0.174419,0.326087,0.128152,0.570789,1.622981,0.454545,Dylan Cease


## 1. Target distribution
Raw K, K/PA rate, and log1p(K).

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df["K"], bins=30); axes[0].set_title("K raw")
axes[1].hist(df["k_rate"], bins=30); axes[1].set_title("K/PA rate")
axes[2].hist(np.log1p(df["K"]), bins=30); axes[2].set_title("log1p(K)")
plt.tight_layout(); plt.savefig(GRAPHS / "k_distribution.png", dpi=120); plt.show()

C:\Users\ckaplinger\AppData\Local\Temp\ipykernel_15168\2258747543.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(GRAPHS / "k_distribution.png", dpi=120); plt.show()


## 2. Correlation with K

In [4]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
corr = df[numeric_cols].corr()["K"].sort_values(ascending=False)
print("Top +correlated with K:"); print(corr.head(20))
print("\nMost -correlated with K:"); print(corr.tail(10))

top_features = corr.abs().sort_values(ascending=False).head(30).index
plt.figure(figsize=(16, 14))
sns.heatmap(df[top_features].corr(), cmap="coolwarm", center=0, annot=False)
plt.title("Top 30 Feature Correlation Matrix")
plt.tight_layout(); plt.savefig(GRAPHS / "corr_heatmap.png", dpi=120); plt.show()

Top +correlated with K:
K             1.000000
k_rate        0.936362
CSW           0.769228
Whiffs        0.745140
Strikes       0.725653
whiff_rate    0.662175
swstr_rate    0.653924
Outs          0.455082
Pitches       0.446997
Swings        0.439769
Chases        0.417418
InZone        0.364704
CS            0.359958
OutZone       0.336151
chase_rate    0.301810
PA            0.295050
ZSwings       0.262267
ff_velo       0.239796
si_velo       0.178808
fc_velo       0.173137
Name: K, dtype: float64

Most -correlated with K:
ff_xwoba        -0.261133
Runs            -0.292628
BIP             -0.303829
wOBA            -0.424359
ocontact_rate   -0.459008
FIP             -0.473896
zcontact_rate   -0.485702
xwOBA           -0.534768
contact_rate    -0.662175
xFIP            -0.755469
Name: K, dtype: float64


C:\Users\ckaplinger\AppData\Local\Temp\ipykernel_15168\1243448731.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(GRAPHS / "corr_heatmap.png", dpi=120); plt.show()


## 3. Skew audit
Flag features that may benefit from a log/transform.

In [5]:
skew_report = {}
for col in numeric_cols:
    if df[col].notna().sum() > 100:
        skew_report[col] = skew(df[col].dropna())
skew_s = pd.Series(skew_report).sort_values(ascending=False)
print("High positive skew (consider log):"); print(skew_s[skew_s > 1])
print("\nHigh negative skew:"); print(skew_s[skew_s < -1])

High positive skew (consider log):
fs_usage_vR    4.118925
fs_usage_vL    3.071520
st_usage_vL    2.935385
rel_z_sd       2.893036
rel_x_sd       2.649489
HBP            2.233609
st_xwoba       2.008715
fs_xwoba       1.985778
cu_xwoba       1.946975
ch_xwoba       1.891689
fs_hb          1.888904
fs_woba        1.886835
st_woba        1.834229
throws_fs      1.814402
fc_usage_vL    1.774137
st_usage_vR    1.717221
cu_woba        1.710014
fc_xwoba       1.709088
sl_xwoba       1.688805
fc_usage_vR    1.668474
sl_woba        1.668011
si_usage_vL    1.630511
si_xwoba       1.626659
ch_woba        1.616492
ff_xwoba       1.496965
ff_woba        1.491231
cu_usage_vR    1.456222
fc_woba        1.439373
FIP            1.389278
si_woba        1.371911
ch_usage_vR    1.316014
sl_usage_vL    1.314910
cu_usage_vL    1.177278
HR             1.173131
si_hb          1.130445
dtype: float64

High negative skew:
pitcher     -1.047204
Pitches     -1.203673
throws_ch   -1.305918
st_hb       -1.396882
t

## 4. Physics features vs K/PA
Scatter with a linear fit for a few key pitch-shape features.

In [6]:
physics_features = [
    "ff_vaa", "ff_ivb", "ff_velo", "ff_spinrate",
    "sl_vaa", "ch_vaa", "cu_vaa",
    "ff_usage_vR", "ff_usage_vL",
]
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, feat in zip(axes.flatten(), physics_features):
    if feat not in df.columns:
        ax.set_visible(False); continue
    sub = df[["k_rate", feat]].dropna()
    ax.scatter(sub[feat], sub["k_rate"], alpha=0.3, s=10)
    ax.set_xlabel(feat); ax.set_ylabel("K/PA")
    if len(sub) > 2:
        m, b = np.polyfit(sub[feat], sub["k_rate"], 1)
        ax.plot(sub[feat], m * sub[feat] + b, color="red", linewidth=1)
plt.tight_layout(); plt.savefig(GRAPHS / "scatter_physics.png", dpi=120); plt.show()

C:\Users\ckaplinger\AppData\Local\Temp\ipykernel_15168\823337708.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(GRAPHS / "scatter_physics.png", dpi=120); plt.show()


## 5. Null audit
Anything above ~10% null is worth flagging for imputation strategy.

In [7]:
null_pct = df.isnull().mean().sort_values(ascending=False)
print("Features >10% null:"); print(null_pct[null_pct > 0.1])
ax = null_pct[null_pct > 0].plot(kind="bar", figsize=(16, 5))
ax.set_title("Null Rate per Feature")
plt.tight_layout(); plt.savefig(GRAPHS / "null_audit.png", dpi=120); plt.show()

Features >10% null:
fs_xwoba       0.855282
fs_woba        0.855282
fs_spinrate    0.836378
fs_hb          0.835953
fs_ivb         0.835953
fs_velo        0.835953
fs_vaa         0.835953
st_xwoba       0.674880
st_woba        0.674880
st_spinrate    0.632116
st_vaa         0.631549
st_hb          0.631549
st_ivb         0.631549
st_velo        0.631549
fc_woba        0.590059
fc_xwoba       0.590059
fc_spinrate    0.538445
fc_vaa         0.537454
fc_ivb         0.537454
fc_hb          0.537454
fc_velo        0.537454
cu_xwoba       0.413197
cu_woba        0.413197
sl_woba        0.385585
sl_xwoba       0.385585
si_xwoba       0.376097
si_woba        0.376097
ch_xwoba       0.328873
ch_woba        0.328873
si_spinrate    0.322713
si_vaa         0.321014
si_hb          0.321014
si_ivb         0.321014
si_velo        0.321014
sl_spinrate    0.319598
sl_vaa         0.317828
sl_velo        0.317828
sl_hb          0.317828
sl_ivb         0.317828
cu_spinrate    0.285047
cu_hb          0.283

C:\Users\ckaplinger\AppData\Local\Temp\ipykernel_15168\444877847.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(GRAPHS / "null_audit.png", dpi=120); plt.show()


## 6. K vs PA (leakage awareness)

`PA` scales `K` almost mechanically, which is exactly why same-game `PA` cannot be
a pregame feature. This plot is a reminder of that relationship, not a feature.

In [8]:
plt.figure(figsize=(8, 5))
plt.scatter(df["PA"], df["K"], alpha=0.3, s=10)
plt.xlabel("PA"); plt.ylabel("K"); plt.title("K vs PA (outcome, not a feature)")
plt.tight_layout(); plt.savefig(GRAPHS / "k_vs_pa.png", dpi=120); plt.show()

C:\Users\ckaplinger\AppData\Local\Temp\ipykernel_15168\3948040598.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(GRAPHS / "k_vs_pa.png", dpi=120); plt.show()


## 7. Stabilization curves

Split-half reliability as a function of window size (starts per half). The dashed
line marks r = 0.70. Uses `Python.reliability.stabilization_curve`.

In [9]:
game_range = list(range(1, 40))

# Single grouped pass with cumulative-sum window slicing (fast).
rel_df = rel.stabilization_curve(df, windows=game_range)

# Figure 1: count/rate stats
fig, ax = plt.subplots(figsize=(14, 6))
for stat in rel.RATE_STATS:
    if stat in rel_df.columns:
        ax.plot(game_range, rel_df[stat], label=stat, linewidth=2)
ax.axhline(0.70, color="black", linestyle="--", linewidth=1, label="r=0.70")
ax.set_xlabel("Starts per half"); ax.set_ylabel("Split-half r")
ax.set_title("Count/Rate Stat Stabilization"); ax.legend(ncol=3, fontsize=8)
plt.tight_layout(); plt.savefig(GRAPHS / "stabilization_counts.png", dpi=120); plt.show()

# Figure 2: pitch physics by pitch type
pitch_groups = {p: [f"{p.lower()}_{s}" for s in ["velo", "spinrate", "ivb", "hb", "vaa"]]
                for p in ["FF", "SI", "SL", "FC", "ST", "CU", "CH"]}
fig, axes = plt.subplots(2, 4, figsize=(22, 10)); axes = axes.flatten()
for ax, (pitch, cols) in zip(axes, pitch_groups.items()):
    for col in cols:
        if col in rel_df.columns:
            ax.plot(game_range, rel_df[col], label=col.split("_", 1)[1], linewidth=1.5)
    ax.axhline(0.70, color="black", linestyle="--", linewidth=1)
    ax.set_title(pitch); ax.set_ylim(0, 1)
    ax.set_xlabel("Starts per half"); ax.set_ylabel("r"); ax.legend(fontsize=7)
axes[-1].set_visible(False)
plt.suptitle("Pitch-Level Stat Stabilization by Pitch Type", fontsize=14)
plt.tight_layout(); plt.savefig(GRAPHS / "stabilization_pitch.png", dpi=120); plt.show()

# Summary: starts-per-half to reach r=0.70
points = {}
for stat in rel_df.columns:
    series = rel_df[stat].dropna()
    crossed = series[series >= 0.70]
    points[stat] = int(crossed.index[0]) if len(crossed) else ">39"
print("Stabilization points (starts per half to reach r=0.70):")
print(pd.Series(points).sort_values(key=lambda s: s.map(lambda v: 999 if v == ">39" else v)).to_string())

C:\Users\ckaplinger\AppData\Local\Temp\ipykernel_15168\2894724428.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(GRAPHS / "stabilization_counts.png", dpi=120); plt.show()


Stabilization points (starts per half to reach r=0.70):
throws_cu          1
throws_ch          1
ff_velo            1
throws_st          1
throws_si          1
ff_spinrate        1
ff_ivb             1
ff_hb              1
throws_sl          1
throws_fc          1
throws_ff          1
cu_ivb             1
ch_hb              1
ch_ivb             1
ch_spinrate        1
ff_usage_vR        1
st_spinrate        1
st_ivb             1
st_hb              1
cu_velo            1
fc_ivb             1
fc_spinrate        1
fc_hb              1
st_velo            1
sl_hb              1
fc_velo            1
sl_spinrate        1
sl_ivb             1
sl_velo            1
si_hb              1
si_ivb             1
si_velo            1
cu_hb              1
cu_spinrate        1
ch_velo            1
rel_x_sd           1
rel_z_sd           1
ch_usage_vL        1
ch_usage_vR        1
cu_usage_vR        1
st_usage_vL        1
st_usage_vR        1
sl_usage_vL        1
sl_usage_vR        1
fc_usage_vL        1

C:\Users\ckaplinger\AppData\Local\Temp\ipykernel_15168\2894724428.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(GRAPHS / "stabilization_pitch.png", dpi=120); plt.show()


## 8. Enhanced reliability table

Split-half Pearson/Spearman, ICC(2,1) (if `pingouin` is installed), year-over-year
correlation, standard error of measurement, and regression slope for each stat.
Uses `Python.reliability.reliability_table`.

In [10]:
metrics_df = rel.reliability_table(df, n_games_per_half=10)
print(f"Enhanced reliability (n=10 starts per half), sorted by YoY r:\n")
cols = ["stat", "pearson_r", "spearman_rho", "icc2", "yoy_r", "sem", "beta1", "feature_tier"]
print(metrics_df[cols].to_string(index=False, float_format=lambda x: f"{x:.3f}"))

Enhanced reliability (n=10 starts per half), sorted by YoY r:

         stat  pearson_r  spearman_rho  icc2  yoy_r    sem  beta1      feature_tier
        si_hb      0.997         0.941   NaN  0.997  0.724  0.995   Tier 1 - stable
        ch_hb      0.996         0.944   NaN  0.994  0.791  0.974   Tier 1 - stable
        rel_x      0.992         0.969   NaN  0.987  0.153  0.979   Tier 1 - stable
        st_hb      0.990         0.925   NaN  0.985  1.096  0.972   Tier 1 - stable
        ff_hb      0.988         0.966   NaN  0.983  0.800  0.984   Tier 1 - stable
        cu_hb      0.988         0.969   NaN  0.980  0.915  0.962   Tier 1 - stable
        rel_z      0.975         0.972   NaN  0.966  0.062  0.993   Tier 1 - stable
    extension      0.974         0.972   NaN  0.961  0.063  0.980   Tier 1 - stable
  cu_spinrate      0.970         0.965   NaN  0.955 42.161  0.987   Tier 1 - stable
  st_spinrate      0.957         0.944   NaN  0.955 48.065  0.984   Tier 1 - stable
  fc_spinrate

## 9. Reliability diagnostics

- **Pearson/Spearman gap > 0.10** -> correlation is outlier-driven.
- **High r but low slope** -> strong regression to the mean.
- **Feature tiers** by year-over-year reliability.

In [11]:
gap = (metrics_df["pearson_r"] - metrics_df["spearman_rho"]).abs()
flagged = metrics_df.assign(gap=gap).query("gap > 0.10").sort_values("gap", ascending=False)
print("Outlier-driven stats (|Pearson - Spearman| > 0.10):")
print(flagged[["stat", "pearson_r", "spearman_rho", "gap"]].to_string(index=False, float_format=lambda x: f"{x:.3f}"))

low_beta = metrics_df.query("pearson_r >= 0.50 and beta1 < 0.60").sort_values("beta1")
print("\nHigh Pearson r but low slope (strong regression-to-mean):")
print(low_beta[["stat", "pearson_r", "beta1", "yoy_r"]].to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\nFeature tiers by YoY reliability:")
for tier in ["Tier 1 - stable", "Tier 2 - moderate", "Tier 3 - noisy", "Unknown"]:
    subset = metrics_df[metrics_df["feature_tier"] == tier]["stat"].tolist()
    print(f"{tier} ({len(subset)}): " + ", ".join(subset))

Outlier-driven stats (|Pearson - Spearman| > 0.10):
     stat  pearson_r  spearman_rho   gap
throws_ff      0.932         0.775 0.157
throws_ch      0.907         0.776 0.131

High Pearson r but low slope (strong regression-to-mean):
         stat  pearson_r  beta1  yoy_r
      Pitches      0.507  0.499  0.567
        Balls      0.537  0.520  0.600
ocontact_rate      0.566  0.522  0.644
  zswing_rate      0.551  0.530  0.553
    zone_rate      0.553  0.552  0.615
      Strikes      0.586  0.553  0.628
   swing_rate      0.606  0.579  0.602
          CSW      0.596  0.585  0.661
       k_rate      0.614  0.599  0.680

Feature tiers by YoY reliability:
Tier 1 - stable (66): si_hb, ch_hb, rel_x, st_hb, ff_hb, cu_hb, rel_z, extension, cu_spinrate, st_spinrate, fc_spinrate, sl_hb, ff_velo, sl_spinrate, ff_spinrate, si_velo, cu_velo, si_ivb, st_velo, ch_usage_vR, ff_ivb, cu_ivb, fc_hb, ch_velo, ff_usage_vR, ch_spinrate, si_spinrate, si_usage_vR, throws_cu, si_vaa, sl_usage_vR, ch_ivb, ff_usa